In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

In [3]:
system_prompt = """

You are a personal chef. The user will give you a list of ingredients they have left over in their house.

or may upload a picutre that has the ingredients left , so you'll have to identify them list them and then

Using the web search tool, search the web for recipes that can be made with the ingredients they have.

Return recipe suggestions and eventually the recipe instructions to the user, if requested.

"""

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from ipywidgets import FileUpload
from IPython.display import display


agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)
uploader = FileUpload(accept='.png', multiple=False)
display(uploader)

FileUpload(value=(), accept='.png', description='Upload')

In [6]:
import base64

# Get the first (and only) uploaded file dict
uploaded_file = uploader.value[0]

# This is a memoryview
content_mv = uploaded_file["content"]

# Convert memoryview -> bytes
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# Now base64 encode
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [7]:
from langchain.messages import HumanMessage

multimodal_question = HumanMessage(content=[
    {"type": "text", "text": "Here are the picture of ingredients i have left"},
    {"type": "image", "base64": img_b64, "mime_type": "image/png"}
])

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [multimodal_question]},
   # {"messages": [HumanMessage(content="I have some leftover lettecue  and chicken tomato sauce . What can I make?")]},
    config
)

print(response['messages'][-1].content)

I see you have eggs, butter, milk, salt, and some sort of white powder (possibly flour or sugar). With these ingredients, you could make a variety of breakfast items or simple baked goods.

Here are a few ideas:

1.  **Scrambled Eggs or Omelette:** A classic and quick option. You can add a splash of milk for creaminess and a pinch of salt.
2.  **Pancakes or Waffles:** If the white powder is flour, you can make a simple pancake or waffle batter with eggs, milk, a little butter, and salt. You might need a leavening agent like baking powder, which isn't visible in the picture.
3.  **Crepes:** Similar to pancakes, crepes are thin and delicate.
4.  **Custard:** With eggs, milk, and sugar (if that's what the white powder is), you can make a simple baked or stovetop custard.

Would you like me to search for a specific recipe, like pancakes or custard? Or would you like me to find a general recipe for one of these options?


In [8]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content=[{'type': 'text', 'text': 'Here are the picture of ingredients i have left'}, {'type': 'image', 'base64': '...', 'mime_type': 'image/png'}], additional_kwargs={}, response_metadata={}, id='569bc1e5-2db1-4ec3-8d0e-520647eed924'),
              AIMessage(content="I see you have eggs, butter, milk, salt, and some sort of white powder (possibly flour or sugar). With these ingredients, you could make a variety of breakfast items or simple baked goods.\n\nHere are a few ideas:\n\n1.  **Scrambled Eggs or Omelette:** A classic and quick option. You can add a splash of milk for creaminess and a pinch of salt.\n2.  **Pancakes or Waffles:** If the white powder is flour, you can make a simple pancake or waffle batter with eggs, milk, a little butter, and salt. You might need a leavening agent like baking powder, which isn't visible in the picture.\n3.  **Crepes:** Similar to pancakes, crepes are thin and delicate.\n4.  **Custard:** With eggs, milk, and sugar (if 